# Kaggle → OpenAI-Compatible Qwen3.8 27B API
### Ollama + Q4_K_M MTP + 2× NVIDIA T4 + optional Cloudflare Quick Tunnel

This notebook is a clean replacement for the original. It removes duplicated recovery cells, conflicting context settings, IPython-only shell cells, fragile DNS polling, and the unauthenticated public proxy.

It provides:

- GPU detection
- Ollama installation
- one persistent model store
- Q4_K_M MTP model pull
- stable model alias `qwen3.8-27b-uncensored-mtp`
- one consistent context configuration
- GPU verification
- native local OpenAI-compatible `/v1`
- authenticated compatibility proxy
- optional temporary Cloudflare HTTPS endpoint
- exact remote `curl` and Python client settings

**Recommended hardware profile:** 2× T4. Default context: **32K**. The model can support a larger context, but maximum model context is not a guarantee that 256K will fit in ~30 GiB of VRAM.

**Context target: 256K (262,144 tokens).** Full native context is configured explicitly; no automatic downgrade is performed.

## 0. Configuration

In [1]:
from pathlib import Path
import os

SOURCE_MODEL = "hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M"
MODEL_NAME = "qwen3.8-27b-uncensored-mtp"

# Full native model context: 262,144 tokens (256K).
CONTEXT_LENGTH = 262144

# MTP draft tokens. The MTP tensors are embedded in the GGUF.
DRAFT_TOKENS = 4

# Quantized KV cache reduces memory pressure at full 256K.
KV_CACHE_TYPE = "q8_0"

OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_MODELS = Path("/kaggle/working/ollama-models")
OLLAMA_LOG = Path("/tmp/ollama-server.log")

PROXY_HOST = "127.0.0.1"
PROXY_PORT = 18000
PROXY_URL = f"http://{PROXY_HOST}:{PROXY_PORT}"

API_KEY = os.environ.get("LLM_API_KEY", "sk-mithu-local")

# Fallback for known Hugging Face/Xet redirect failures.
ALLOW_HF_INSECURE_FALLBACK = True

OLLAMA_MODELS.mkdir(parents=True, exist_ok=True)

print("Source model :", SOURCE_MODEL)
print("Model name   :", MODEL_NAME)
print("Context      :", f"{CONTEXT_LENGTH:,} tokens (256K)")
print("KV cache     :", KV_CACHE_TYPE)
print("Draft tokens :", DRAFT_TOKENS)
print("Model store  :", OLLAMA_MODELS)
print("Proxy        :", PROXY_URL)
print("API key      :", API_KEY)


Source model : hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M
Model name   : qwen3.8-27b-uncensored-mtp
Context      : 262,144 tokens (256K)
KV cache     : q8_0
Draft tokens : 4
Model store  : /kaggle/working/ollama-models
Proxy        : http://127.0.0.1:18000
API key      : sk-mithu-local


## 1. Check the Kaggle GPU profile for 256K

In [2]:
import subprocess
import re

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    raise RuntimeError(
        "NVIDIA GPU not detected. Enable Kaggle GPU acceleration first.\n"
        + result.stderr
    )

gpu_lines = [x.strip() for x in result.stdout.splitlines() if x.strip()]

print("=" * 72)
print("GPU DETECTION FOR 256K DEPLOYMENT")
print("=" * 72)
print("\n".join(gpu_lines))

gpu_count = len(gpu_lines)
print(f"\nGPU count: {gpu_count}")

if gpu_count < 2:
    print("⚠️ This notebook targets 2×T4 (16 GiB each).")
else:
    print("✅ Multi-GPU profile detected.")

total_gib = 0.0
for line in gpu_lines:
    m = re.search(r"([0-9.]+)\s*MiB", line)
    if m:
        total_gib += float(m.group(1)) / 1024.0

print(f"Total reported VRAM: ~{total_gib:.1f} GiB")
print("256K mode uses Q8_0 KV cache to reduce memory pressure.")
print("Model file is ~16.8 GB Q4_K_M; runtime also needs KV cache and overhead.")


GPU DETECTION FOR 256K DEPLOYMENT
0, Tesla T4, 15360 MiB
1, Tesla T4, 15360 MiB

GPU count: 2
✅ Multi-GPU profile detected.
Total reported VRAM: ~30.0 GiB
256K mode uses Q8_0 KV cache to reduce memory pressure.
Model file is ~16.8 GB Q4_K_M; runtime also needs KV cache and overhead.


## 2. Install Ollama

In [3]:
import os
import shutil
import subprocess

def run_cmd(cmd, *, check=True):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, check=check)

OLLAMA_BIN = shutil.which("ollama")

if not OLLAMA_BIN:
    # The current Ollama installer extracts a compressed archive with zstd.
    # Kaggle images may not have zstd preinstalled, so install it explicitly first.
    missing = []
    if shutil.which("zstd") is None:
        missing.append("zstd")
    if shutil.which("curl") is None:
        missing.append("curl")

    if missing:
        print("Installing required OS packages:", ", ".join(missing))
        run_cmd(["apt-get", "update", "-qq"])
        run_cmd(["bash", "-lc", "DEBIAN_FRONTEND=noninteractive apt-get install -y -qq " + " ".join(missing)])

    # Verify the extraction dependency before invoking the installer.
    if shutil.which("zstd") is None:
        raise RuntimeError(
            "zstd is still unavailable after apt installation. "
            "Run: apt-get update && apt-get install -y zstd"
        )

    print("✅ zstd:", shutil.which("zstd"))
    print("Installing Ollama to /usr/local ...")

    install = subprocess.run(
        ["bash", "-lc", "curl -fsSL https://ollama.com/install.sh | sh"],
        text=True,
    )

    if install.returncode != 0:
        raise RuntimeError(
            "Ollama installation failed even after installing zstd. "
            "See the installer output above."
        )

    OLLAMA_BIN = shutil.which("ollama") or "/usr/local/bin/ollama"

if not os.path.exists(OLLAMA_BIN):
    raise RuntimeError(f"Ollama executable was not found: {OLLAMA_BIN}")

version = subprocess.run(
    [OLLAMA_BIN, "--version"],
    capture_output=True,
    text=True,
    check=True,
)

print("✅ Ollama:", version.stdout.strip() or version.stderr.strip())
print("Binary   :", OLLAMA_BIN)


Installing required OS packages: zstd
$ apt-get update -qq


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


$ bash -lc DEBIAN_FRONTEND=noninteractive apt-get install -y -qq zstd
Selecting previously unselected package zstd.
(Reading database ... 125186 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
✅ zstd: /usr/bin/zstd
Installing Ollama to /usr/local ...


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...


✅ Ollama: Warning: could not connect to a running Ollama instance
Binary   : /usr/local/bin/ollama


>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


## 3. Start one clean Ollama server

The original notebook configured 256K in one cell and later restarted Ollama at 32K. This replacement has one consistent context value and one server-start path.

In [4]:
import os
import subprocess
import time
import urllib.request

OLLAMA_ENV = os.environ.copy()
OLLAMA_ENV.update({
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_MODELS": str(OLLAMA_MODELS),
    "OLLAMA_CONTEXT_LENGTH": str(CONTEXT_LENGTH),
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": KV_CACHE_TYPE,
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_KEEP_ALIVE": "-1",
    "OLLAMA_SCHED_SPREAD": "2",
})

def ollama_ready():
    try:
        with urllib.request.urlopen(f"{OLLAMA_URL}/api/tags", timeout=2) as r:
            return r.status == 200
    except Exception:
        return False

old_process = globals().get("ollama_server_process")
if old_process is not None and old_process.poll() is None:
    old_process.terminate()
    try:
        old_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_process.kill()

if ollama_ready():
    subprocess.run(
        ["pkill", "-f", "ollama serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(2)

server_log = open(OLLAMA_LOG, "w", buffering=1)

ollama_server_process = subprocess.Popen(
    [OLLAMA_BIN, "serve"],
    env=OLLAMA_ENV,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

deadline = time.monotonic() + 60
while time.monotonic() < deadline:
    if ollama_ready():
        break
    if ollama_server_process.poll() is not None:
        break
    time.sleep(1)

if not ollama_ready():
    tail = OLLAMA_LOG.read_text(encoding="utf-8", errors="ignore")[-12000:]
    raise RuntimeError(f"Ollama did not become ready. Log: {OLLAMA_LOG}\n\n{tail}")

print("✅ Ollama is ready")
print("URL            :", OLLAMA_URL)
print("Context        :", f"{CONTEXT_LENGTH:,} tokens (256K)")
print("Flash Attention:", OLLAMA_ENV["OLLAMA_FLASH_ATTENTION"])
print("KV cache       :", OLLAMA_ENV["OLLAMA_KV_CACHE_TYPE"])
print("Parallel       :", OLLAMA_ENV["OLLAMA_NUM_PARALLEL"])
print("GPU scheduling :", OLLAMA_ENV["OLLAMA_SCHED_SPREAD"])

✅ Ollama is ready
URL            : http://127.0.0.1:11434
Context        : 262,144 tokens (256K)
Flash Attention: 1
KV cache       : q8_0
Parallel       : 1
GPU scheduling : 2


## 4. Pull the uncensored Qwen3.8 Q4_K_M MTP model

The selected Hugging Face repository documents direct Ollama use and includes a Q4_K_M MTP GGUF.

In [5]:
import subprocess

print("=" * 72)
print("PULLING MODEL")
print("=" * 72)
print(SOURCE_MODEL)

pull = subprocess.run(
    [OLLAMA_BIN, "pull", SOURCE_MODEL],
    env=OLLAMA_ENV,
)

if pull.returncode != 0:
    if not ALLOW_HF_INSECURE_FALLBACK:
        raise RuntimeError(
            "Normal `ollama pull` failed. Set ALLOW_HF_INSECURE_FALLBACK=True "
            "only for the known Hugging Face/Xet redirect case."
        )

    print("\n⚠️ Normal pull failed; retrying with --insecure.")
    subprocess.run(
        [OLLAMA_BIN, "pull", "--insecure", SOURCE_MODEL],
        env=OLLAMA_ENV,
        check=True,
    )

print("\n✅ Model is available.")
subprocess.run([OLLAMA_BIN, "list"], env=OLLAMA_ENV, check=True)

PULLING MODEL
hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M


pulling manifest ⠋ pulling manifest ⠙ pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏ 1.6 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏ 6.8 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   0% ▕                  ▏  59 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   1% ▕                  ▏ 119 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   1% ▕                  ▏ 149 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   1% ▕                  ▏ 213 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   2% ▕                  ▏ 273 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   2% ▕                  ▏ 302 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   2% ▕                  ▏ 362 MB/ 16 GB                  pulling manifest 
pulling 4c5e2db039e9:   3% ▕                  ▏ 420 MB/ 16 GB          


✅ Model is available.
NAME                                                        ID              SIZE     MODIFIED               
hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M    2859adf8326a    17 GB    Less than a second ago    


pulling manifest 
pulling 4c5e2db039e9: 100% ▕█████████████████ ▏  16 GB/ 16 GB   91 MB/s      0s
pulling 5ac423f8a290:  98% ▕█████████████████ ▏ 906 MB/927 MB  413 MB/s      0s
verifying sha256 digest ⠹ pulling manifest 
pulling 4c5e2db039e9: 100% ▕█████████████████ ▏  16 GB/ 16 GB   91 MB/s      0s
pulling 5ac423f8a290:  98% ▕█████████████████ ▏ 906 MB/927 MB  413 MB/s      0s
verifying sha256 digest 
writing manifest 
success 


CompletedProcess(args=['/usr/local/bin/ollama', 'list'], returncode=0)

## 5. Create the stable model alias

For OpenAI-compatible clients, Ollama recommends fixing the context in a Modelfile because the OpenAI API itself does not expose `num_ctx`.

In [6]:
from pathlib import Path
import subprocess

modelfile = Path("/kaggle/working/Qwen38-Uncensored-MTP-256K.Modelfile")
modelfile.write_text(
    f"FROM {SOURCE_MODEL}\n"
    f"PARAMETER num_ctx {CONTEXT_LENGTH}\n"
    f"PARAMETER draft_num_predict {DRAFT_TOKENS}\n",
    encoding="utf-8",
)

print(modelfile.read_text(encoding="utf-8"))

subprocess.run(
    [OLLAMA_BIN, "create", MODEL_NAME, "-f", str(modelfile)],
    env=OLLAMA_ENV,
    check=True,
)

print(f"✅ Created {MODEL_NAME} with num_ctx={CONTEXT_LENGTH:,} (256K)")
subprocess.run([OLLAMA_BIN, "show", MODEL_NAME], env=OLLAMA_ENV, check=True)


FROM hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M
PARAMETER num_ctx 262144
PARAMETER draft_num_predict 4

✅ Created qwen3.8-27b-uncensored-mtp with num_ctx=262,144 (256K)
  Model
    architecture        qwen35    
    parameters          27.3B     
    context length      262144    
    embedding length    5120      
    quantization        Q4_K_M    

  Capabilities
    tools         
    thinking      
    completion    
    vision        

  Projector
    architecture        clip       
    parameters          460.73M    
    embedding length    1152       
    dimensions          5120       

  Parameters
    num_ctx              262144    
    draft_num_predict    4         



gathering model components 
using existing layer sha256:4c5e2db039e9325ac7724c8846c71356a24ad1cdfa28002d73ecb6be645f9675 
using existing layer sha256:5ac423f8a29059dc24e51bc6a43e9380dcd57a9347f28b62591e0b3f60b7081c 
creating new layer sha256:785b33c989b705af3cf004bad92089702ac09e49a7a08eae259de94c09a212ea 
writing manifest 
success 


CompletedProcess(args=['/usr/local/bin/ollama', 'show', 'qwen3.8-27b-uncensored-mtp'], returncode=0)

## 6. Warm up the model and verify GPU execution

The warm-up request explicitly uses `num_ctx=262144` to verify the requested 256K window.

In [ ]:
import json
import subprocess
import time
import urllib.error
import urllib.request

payload = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Reply with exactly: MODEL IS WORKING"}],
    "stream": False,
    "keep_alive": -1,
    "options": {
        "num_ctx": CONTEXT_LENGTH,
        "temperature": 0,
    },
}

request = urllib.request.Request(
    f"{OLLAMA_URL}/api/chat",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)

start = time.perf_counter()

try:
    with urllib.request.urlopen(request, timeout=900) as response:
        result = json.loads(response.read())
except urllib.error.HTTPError as exc:
    body = exc.read().decode("utf-8", errors="ignore")
    raise RuntimeError(
        f"256K model load/inference failed with HTTP {exc.code}.\n"
        f"Server response:\n{body}\n\n"
        "The notebook does not silently downgrade. Inspect the error and GPU memory."
    ) from exc
except Exception as exc:
    log_tail = OLLAMA_LOG.read_text(encoding="utf-8", errors="ignore")[-16000:]
    raise RuntimeError(
        "256K model load/inference failed.\n"
        f"{type(exc).__name__}: {exc}\n\n"
        f"Recent Ollama log:\n{log_tail}"
    ) from exc

elapsed = time.perf_counter() - start
print("Response :", result.get("message", {}).get("content", "").strip())
print("Wall time:", f"{elapsed:.2f}s")

ps = subprocess.run(
    [OLLAMA_BIN, "ps"],
    env=OLLAMA_ENV,
    capture_output=True,
    text=True,
    check=True,
)

print("\n" + "=" * 72)
print("OLLAMA PS — 256K LOAD CHECK")
print("=" * 72)
print(ps.stdout)

if "GPU" in ps.stdout:
    print("✅ GPU execution detected.")
else:
    print("⚠️ No GPU processor entry found; inspect the log.")

print("\nNVIDIA MEMORY")
subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.used,memory.total", "--format=csv"],
    check=True,
)


## 7. Test Ollama's native OpenAI-compatible API locally

In [ ]:
import json
import urllib.request

headers = {
    "Authorization": "Bearer ollama",
    "Content-Type": "application/json",
}

with urllib.request.urlopen(
    urllib.request.Request(
        f"{OLLAMA_URL}/v1/models",
        headers=headers,
        method="GET",
    ),
    timeout=20,
) as response:
    models = json.loads(response.read())

print("Models visible through /v1/models:")
for item in models.get("data", []):
    print("  •", item.get("id"))

payload = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Reply with exactly: LOCAL API WORKS"}],
    "stream": False,
    "temperature": 0,
    "max_tokens": 32,
    "options": {"num_ctx": CONTEXT_LENGTH},
}

request = urllib.request.Request(
    f"{OLLAMA_URL}/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers=headers,
    method="POST",
)

with urllib.request.urlopen(request, timeout=180) as response:
    result = json.loads(response.read())

print("\nResponse:", result["choices"][0]["message"]["content"])
print("✅ Native local OpenAI-compatible API works.")
print(f"Requested context: {CONTEXT_LENGTH:,} tokens (256K)")


## 8. Optional benchmark

In [9]:
import json
import time
import urllib.request

prompt = (
    "Explain in about 120 words why GPUs are useful for large language models. "
    "Mention memory bandwidth, parallel computation, and matrix multiplication."
)

payload = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": prompt}],
    "stream": False,
    "temperature": 0,
    "max_tokens": 180,
    "options": {"num_ctx": CONTEXT_LENGTH},
}

request = urllib.request.Request(
    f"{OLLAMA_URL}/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers={"Authorization": "Bearer ollama", "Content-Type": "application/json"},
    method="POST",
)

start = time.perf_counter()
with urllib.request.urlopen(request, timeout=600) as response:
    result = json.loads(response.read())

wall = time.perf_counter() - start
usage = result.get("usage", {})
tokens = usage.get("completion_tokens")

print(result["choices"][0]["message"]["content"])
print("\nWall time:", f"{wall:.2f}s")
print("Completion tokens:", tokens)
if tokens:
    print("Approx wall tok/s:", f"{tokens / wall:.2f}")
print(f"Context requested: {CONTEXT_LENGTH:,} tokens (256K)")




Wall time: 104.59s
Completion tokens: 180
Approx wall tok/s: 1.72
Context requested: 262,144 tokens (256K)


## 9. Install Cloudflare `cloudflared`

In [10]:
import shutil
import subprocess

if not shutil.which("cloudflared"):
    deb = "/tmp/cloudflared-linux-amd64.deb"
    subprocess.run(
        [
            "bash",
            "-lc",
            'curl -fL "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb" '
            f'-o "{deb}"',
        ],
        check=True,
    )
    subprocess.run(["dpkg", "-i", deb], check=False)
    subprocess.run(["apt-get", "install", "-f", "-y", "-qq"], check=True)

print(
    subprocess.run(
        ["cloudflared", "--version"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
)

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 18.2M  100 18.2M    0     0  42.8M      0 --:--:-- --:--:-- --:--:-- 42.8M


Selecting previously unselected package cloudflared.
(Reading database ... 125207 files and directories currently installed.)
Preparing to unpack .../cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.1) ...
Setting up cloudflared (2026.9.1) ...
Processing triggers for man-db (2.10.2-1) ...
cloudflared version 2026.9.1 (built 2026-09-11-13:35 UTC)


## 10. Start the authenticated compatibility proxy

Ollama already supports `/v1`, so this proxy is not required for local use. It fixes the public-facing problems in the original notebook by adding bearer-key validation and preserving the Qwen3.8 message-order normalization.

In [11]:
from pathlib import Path
import os
import subprocess
import sys
import time
import urllib.request

PROXY_SCRIPT = '\nimport json\nimport os\n\nimport httpx\nimport uvicorn\nfrom fastapi import FastAPI, Request\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import JSONResponse, Response, StreamingResponse\n\napp = FastAPI(title="Kaggle Qwen3.8 OpenAI Proxy")\n\napp.add_middleware(\n    CORSMiddleware,\n    allow_origins=["*"],\n    allow_credentials=False,\n    allow_methods=["*"],\n    allow_headers=["*"],\n)\n\nOLLAMA_BASE = os.environ.get("OLLAMA_BASE", "http://127.0.0.1:11434")\nAPI_KEY = os.environ.get("LLM_API_KEY", "sk-mithu-local")\n\nHOP_BY_HOP = {\n    "connection", "keep-alive", "proxy-authenticate",\n    "proxy-authorization", "te", "trailer",\n    "transfer-encoding", "upgrade", "content-length",\n}\n\ndef text_content(content):\n    if isinstance(content, str):\n        return content\n    if content is None:\n        return ""\n    if isinstance(content, list):\n        parts = []\n        for item in content:\n            if isinstance(item, str):\n                parts.append(item)\n            elif isinstance(item, dict) and item.get("type") in {"text", "input_text"}:\n                parts.append(str(item.get("text", "")))\n        return "\\n".join(parts)\n    return str(content)\n\ndef normalize_messages(messages):\n    if not isinstance(messages, list):\n        return messages\n\n    system_parts = []\n    normal = []\n\n    for msg in messages:\n        if not isinstance(msg, dict):\n            normal.append(msg)\n            continue\n\n        role = msg.get("role")\n        if role in {"system", "developer"}:\n            content = text_content(msg.get("content"))\n            if content.strip():\n                system_parts.append(content.strip())\n        else:\n            normal.append(msg)\n\n    if not system_parts:\n        return normal\n\n    return [{"role": "system", "content": "\\n\\n".join(system_parts)}] + normal\n\ndef bearer_token(request: Request):\n    value = request.headers.get("authorization", "")\n    if not value.lower().startswith("bearer "):\n        return None\n    return value[7:].strip()\n\n@app.get("/healthz")\nasync def healthz():\n    return {"status": "ok"}\n\n@app.api_route(\n    "/{path:path}",\n    methods=["GET", "POST", "PUT", "PATCH", "DELETE", "HEAD", "OPTIONS"],\n)\nasync def proxy(path: str, request: Request):\n    if not request.url.path.startswith("/v1/"):\n        return JSONResponse(\n            {"error": {"message": "Only /v1/* endpoints are exposed."}},\n            status_code=404,\n        )\n\n    if bearer_token(request) != API_KEY:\n        return JSONResponse(\n            {\n                "error": {\n                    "message": "Invalid or missing bearer API key.",\n                    "type": "authentication_error",\n                }\n            },\n            status_code=401,\n        )\n\n    body = await request.body()\n\n    headers = {\n        k: v\n        for k, v in request.headers.items()\n        if k.lower() not in HOP_BY_HOP\n        and k.lower() not in {"host", "authorization"}\n    }\n\n    is_stream = False\n\n    if body and request.method in {"POST", "PUT", "PATCH"}:\n        try:\n            payload = json.loads(body.decode("utf-8"))\n        except Exception:\n            payload = None\n\n        if isinstance(payload, dict):\n            if request.url.path == "/v1/chat/completions":\n                if "messages" in payload:\n                    payload["messages"] = normalize_messages(payload["messages"])\n\n                # Guarantee the configured full-context default for standard OpenAI clients.\n                options = payload.get("options")\n                if not isinstance(options, dict):\n                    options = {}\n                    payload["options"] = options\n                options.setdefault("num_ctx", 262144)\n\n            is_stream = bool(payload.get("stream", False))\n            body = json.dumps(payload, ensure_ascii=False).encode("utf-8")\n            headers["content-type"] = "application/json"\n\n    target = OLLAMA_BASE + request.url.path\n\n    async with httpx.AsyncClient(timeout=None) as client:\n        if is_stream and request.method == "POST":\n            upstream_cm = client.stream(\n                request.method,\n                target,\n                content=body,\n                headers=headers,\n                params=request.query_params,\n            )\n            upstream = await upstream_cm.__aenter__()\n\n            async def iterator():\n                try:\n                    async for chunk in upstream.aiter_bytes():\n                        yield chunk\n                finally:\n                    await upstream_cm.__aexit__(None, None, None)\n\n            out_headers = {\n                k: v\n                for k, v in upstream.headers.items()\n                if k.lower() not in HOP_BY_HOP\n            }\n\n            return StreamingResponse(\n                iterator(),\n                status_code=upstream.status_code,\n                headers=out_headers,\n                media_type=upstream.headers.get("content-type"),\n            )\n\n        upstream = await client.request(\n            request.method,\n            target,\n            content=body,\n            headers=headers,\n            params=request.query_params,\n        )\n\n        out_headers = {\n            k: v\n            for k, v in upstream.headers.items()\n            if k.lower() not in HOP_BY_HOP\n        }\n\n        return Response(\n            content=upstream.content,\n            status_code=upstream.status_code,\n            headers=out_headers,\n            media_type=upstream.headers.get("content-type"),\n        )\n\nif __name__ == "__main__":\n    uvicorn.run(app, host="127.0.0.1", port=18000, log_level="info")\n'

proxy_file = Path("/kaggle/working/qwen38_openai_proxy.py")
proxy_file.write_text(PROXY_SCRIPT, encoding="utf-8")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn", "httpx"],
    check=True,
)

proxy_env = os.environ.copy()
proxy_env.update({
    "OLLAMA_BASE": OLLAMA_URL,
    "LLM_API_KEY": API_KEY,
})

old_proxy = globals().get("qwen38_proxy_process")
if old_proxy is not None and old_proxy.poll() is None:
    old_proxy.terminate()
    try:
        old_proxy.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_proxy.kill()

proxy_log = open("/tmp/qwen38-proxy.log", "w", buffering=1)

qwen38_proxy_process = subprocess.Popen(
    [sys.executable, str(proxy_file)],
    env=proxy_env,
    stdout=proxy_log,
    stderr=subprocess.STDOUT,
)

deadline = time.monotonic() + 30
while time.monotonic() < deadline:
    try:
        with urllib.request.urlopen(f"{PROXY_URL}/healthz", timeout=2) as response:
            if response.status == 200:
                break
    except Exception:
        pass

    if qwen38_proxy_process.poll() is not None:
        break
    time.sleep(1)
else:
    log_tail = Path("/tmp/qwen38-proxy.log").read_text(
        encoding="utf-8", errors="ignore"
    )[-8000:]
    raise RuntimeError(f"Proxy did not start.\n\n{log_tail}")

print("✅ Proxy ready:", PROXY_URL)
print("Bearer key protection:", "enabled")

✅ Proxy ready: http://127.0.0.1:18000
Bearer key protection: enabled


## 11. Test the proxy locally

In [12]:
import json
import urllib.request

headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
}

with urllib.request.urlopen(
    urllib.request.Request(
        f"{PROXY_URL}/v1/models",
        headers=headers,
        method="GET",
    ),
    timeout=20,
) as response:
    models = json.loads(response.read())

print("Models:", [x.get("id") for x in models.get("data", [])])

payload = {
    "model": MODEL_NAME,
    "messages": [
        {"role": "user", "content": "Say exactly: PROXY API WORKS"},
        {"role": "developer", "content": "Be concise."},
        {"role": "system", "content": "Follow the user request exactly."},
    ],
    "stream": False,
    "temperature": 0,
    "max_tokens": 32,
}

request = urllib.request.Request(
    f"{PROXY_URL}/v1/chat/completions",
    data=json.dumps(payload).encode("utf-8"),
    headers=headers,
    method="POST",
)

with urllib.request.urlopen(request, timeout=180) as response:
    result = json.loads(response.read())

print("Response:", result["choices"][0]["message"]["content"])
print("✅ Proxy, bearer auth, and message normalization work.")

Models: ['qwen3.8-27b-uncensored-mtp:latest', 'hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M']
Response: 
✅ Proxy, bearer auth, and message normalization work.


## 12. Start the Cloudflare Quick Tunnel

In [13]:
import re
import subprocess
import time
from pathlib import Path

old_tunnel = globals().get("cloudflared_process")
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()

subprocess.run(
    ["pkill", "-f", "cloudflared tunnel --url"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(1)

tunnel_log_path = Path("/tmp/cloudflared-qwen38.log")
tunnel_log = open(tunnel_log_path, "w", buffering=1)

cloudflared_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--no-autoupdate",
        "--protocol", "http2",
        "--edge-ip-version", "4",
        "--url", PROXY_URL,
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

PUBLIC_OLLAMA_URL = None
deadline = time.monotonic() + 60

while time.monotonic() < deadline:
    log_text = tunnel_log_path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    match = re.search(
        r"https://[a-z0-9-]+\.trycloudflare\.com",
        log_text,
        re.IGNORECASE,
    )

    if match:
        PUBLIC_OLLAMA_URL = match.group(0).rstrip("/")
        break

    if cloudflared_process.poll() is not None:
        break

    time.sleep(1)

if not PUBLIC_OLLAMA_URL:
    details = tunnel_log_path.read_text(
        encoding="utf-8", errors="ignore"
    )[-12000:]
    raise RuntimeError(
        "Cloudflare Quick Tunnel did not produce a public URL.\n\n"
        + details
    )

OPENAI_COMPAT_BASE_URL = f"{PUBLIC_OLLAMA_URL}/v1"

print("=" * 72)
print("CLOUDFLARE QUICK TUNNEL READY")
print("=" * 72)
print("PID       :", cloudflared_process.pid)
print("Public URL:", PUBLIC_OLLAMA_URL)
print("Base URL  :", OPENAI_COMPAT_BASE_URL)
print("Model     :", MODEL_NAME)
print("API key   :", API_KEY)

# Do not let later cells use a dead or stale Quick Tunnel.
if cloudflared_process.poll() is not None:
    raise RuntimeError(
        f"cloudflared exited unexpectedly with code {cloudflared_process.returncode}."
    )

from urllib.parse import urlparse
from socket import getaddrinfo, gaierror, SOCK_STREAM

host = urlparse(PUBLIC_OLLAMA_URL).hostname
print(f"Tunnel hostname: {host}")
try:
    addresses = sorted({item[4][0] for item in getaddrinfo(host, 443, type=SOCK_STREAM)})
    print("✅ Kaggle DNS resolved the tunnel hostname:", addresses)
except gaierror as exc:
    print("⚠️ Kaggle system DNS cannot resolve the Quick Tunnel hostname.")
    print("DNS error:", exc)
    print("This is separate from Ollama and 256K context.")
    print("Copy PUBLIC_OLLAMA_URL and test it from Ubuntu/phone instead.")


CLOUDFLARE QUICK TUNNEL READY
PID       : 1202
Public URL: https://contests-revisions-figure-infants.trycloudflare.com
Base URL  : https://contests-revisions-figure-infants.trycloudflare.com/v1
Model     : qwen3.8-27b-uncensored-mtp
API key   : sk-mithu-local
Tunnel hostname: contests-revisions-figure-infants.trycloudflare.com
⚠️ Kaggle system DNS cannot resolve the Quick Tunnel hostname.
DNS error: [Errno -2] Name or service not known
This is separate from Ollama and 256K context.
Copy PUBLIC_OLLAMA_URL and test it from Ubuntu/phone instead.


## 13. Public smoke test

Use non-streaming requests.

Cloudflare's current Quick Tunnel documentation states that Quick Tunnels do not support Server-Sent Events (SSE), so streaming through the temporary public tunnel is not the reliable path.

In [14]:
import json
import urllib.request
from urllib.parse import urlparse
from socket import getaddrinfo, gaierror, SOCK_STREAM

def public_get(path, timeout=45):
    if not PUBLIC_OLLAMA_URL:
        raise RuntimeError("PUBLIC_OLLAMA_URL is empty. Re-run the Cloudflare tunnel cell.")
    if 'cloudflared_process' in globals() and cloudflared_process.poll() is not None:
        raise RuntimeError("cloudflared is no longer running. Re-run the tunnel cell to get a fresh URL.")

    host = urlparse(PUBLIC_OLLAMA_URL).hostname
    try:
        getaddrinfo(host, 443, type=SOCK_STREAM)
    except gaierror as exc:
        print("⚠️ Kaggle DNS resolution failed:", exc)
        print("Public URL:", PUBLIC_OLLAMA_URL)
        print("This does NOT prove the tunnel is down.")
        print("Test the URL from Ubuntu/phone on another network.")
        return None

    request = urllib.request.Request(
        f"{PUBLIC_OLLAMA_URL}{path}",
        headers={"Authorization": f"Bearer {API_KEY}"},
        method="GET",
    )
    with urllib.request.urlopen(request, timeout=timeout) as response:
        return json.loads(response.read())

try:
    body = public_get("/v1/models")
    if body is not None:
        print("✅ Public /v1/models works")
        print(json.dumps(body, indent=2)[:5000])
except Exception as exc:
    print("⚠️ Public /v1/models request failed:", type(exc).__name__, exc)


⚠️ Kaggle DNS resolution failed: [Errno -2] Name or service not known
Public URL: https://contests-revisions-figure-infants.trycloudflare.com
This does NOT prove the tunnel is down.
Test the URL from Ubuntu/phone on another network.


In [15]:
import json
import urllib.request
from urllib.parse import urlparse
from socket import getaddrinfo, gaierror, SOCK_STREAM

payload = {
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Reply with exactly: PUBLIC 256K API WORKS"}],
    "stream": False,
    "max_tokens": 32,
    "options": {"num_ctx": CONTEXT_LENGTH},
}

try:
    if not PUBLIC_OLLAMA_URL:
        raise RuntimeError("PUBLIC_OLLAMA_URL is empty. Re-run the Cloudflare tunnel cell.")
    if 'cloudflared_process' in globals() and cloudflared_process.poll() is not None:
        raise RuntimeError("cloudflared is no longer running. Re-run the tunnel cell.")

    host = urlparse(PUBLIC_OLLAMA_URL).hostname
    try:
        getaddrinfo(host, 443, type=SOCK_STREAM)
    except gaierror as exc:
        print("⚠️ Kaggle cannot resolve the Cloudflare hostname:", exc)
        print("PUBLIC_OLLAMA_URL:", PUBLIC_OLLAMA_URL)
        print("Use the URL from another device to test the public API.")
        raise SystemExit

    request = urllib.request.Request(
        f"{PUBLIC_OLLAMA_URL}/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=900) as response:
        result = json.loads(response.read())

    print("Response:", result["choices"][0]["message"]["content"])
    print("✅ Public OpenAI-compatible API reached the 256K-configured proxy.")
    print(f"Requested context: {CONTEXT_LENGTH:,} tokens")
except SystemExit:
    pass
except Exception as exc:
    print("⚠️ Public 256K API smoke test failed:", type(exc).__name__, exc)
    print("Check the Cloudflare tunnel and local proxy first.")


⚠️ Kaggle cannot resolve the Cloudflare hostname: [Errno -2] Name or service not known
PUBLIC_OLLAMA_URL: https://contests-revisions-figure-infants.trycloudflare.com
Use the URL from another device to test the public API.


## 14. Exact settings for another device

In [16]:
print("=" * 72)
print("COPY THESE SETTINGS — 256K")
print("=" * 72)
print("\nContext: 256K (262,144 tokens)")

print("\nBase URL:")
print(OPENAI_COMPAT_BASE_URL)

print("\nModel:")
print(MODEL_NAME)

print("\nAPI key:")
print(API_KEY)

print("\nOpenAI Python SDK:")
python_lines = [
    "from openai import OpenAI",
    "",
    "client = OpenAI(",
    f'    base_url="{OPENAI_COMPAT_BASE_URL}/",',
    f'    api_key="{API_KEY}",',
    ")",
    "",
    "response = client.chat.completions.create(",
    f'    model="{MODEL_NAME}",',
    '    messages=[{"role": "user", "content": "Hello from my device"}],',
    "    stream=False,",
    "    max_tokens=256,",
    ")",
    "",
    "print(response.choices[0].message.content)",
]
print("\n".join(python_lines))

print("\ncurl:")
curl = (
    f'curl -sS -H "Authorization: Bearer {API_KEY}" '
    f'-H "Content-Type: application/json" '
    f'"{OPENAI_COMPAT_BASE_URL}/chat/completions" '
    f'-d \'{{"model":"{MODEL_NAME}","messages":[{{"role":"user","content":"Reply with exactly: REMOTE API WORKS"}}],"stream":false,"max_tokens":64}}\''
)
print(curl)

COPY THESE SETTINGS — 256K

Context: 256K (262,144 tokens)

Base URL:
https://contests-revisions-figure-infants.trycloudflare.com/v1

Model:
qwen3.8-27b-uncensored-mtp

API key:
sk-mithu-local

OpenAI Python SDK:
from openai import OpenAI

client = OpenAI(
    base_url="https://contests-revisions-figure-infants.trycloudflare.com/v1/",
    api_key="sk-mithu-local",
)

response = client.chat.completions.create(
    model="qwen3.8-27b-uncensored-mtp",
    messages=[{"role": "user", "content": "Hello from my device"}],
    stream=False,
    max_tokens=256,
)

print(response.choices[0].message.content)

curl:
curl -sS -H "Authorization: Bearer sk-mithu-local" -H "Content-Type: application/json" "https://contests-revisions-figure-infants.trycloudflare.com/v1/chat/completions" -d '{"model":"qwen3.8-27b-uncensored-mtp","messages":[{"role":"user","content":"Reply with exactly: REMOTE API WORKS"}],"stream":false,"max_tokens":64}'


## 15. Optional local chat helper

In [17]:
import json
import urllib.request

def chat(prompt: str, system: str | None = None, base_url: str = PROXY_URL) -> str:
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "stream": False,
        "keep_alive": -1,
        "max_tokens": 512,
    }

    request = urllib.request.Request(
        f"{base_url.rstrip('/')}/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        method="POST",
    )

    with urllib.request.urlopen(request, timeout=600) as response:
        result = json.loads(response.read())

    return result["choices"][0]["message"]["content"]

print(chat("Write one line of Python that prints hello."))

`print("hello")`


## 16. Troubleshooting 256K

**OOM / failed model load:** 256K is the intended configuration. Check `ollama ps`, `nvidia-smi`, and `/tmp/ollama-server.log`.

**No silent fallback:** this notebook deliberately stops with a diagnostic error rather than changing 256K to a smaller context automatically.

**Memory:** the model is ~16.8 GB Q4_K_M, and full 256K additionally requires KV-cache and runtime memory. Q8_0 KV cache is enabled here to reduce memory pressure.

**Latency:** very long-context prefill can be much slower than short prompts even when the model fits.

**Manual fallback only:** if your selected Kaggle runtime cannot fit 256K, change `CONTEXT_LENGTH` to `131072` or `65536`, then rerun the server/model creation cells.


## 17. Optional Ollama CLI

In [18]:
import subprocess
subprocess.run([OLLAMA_BIN, "run", MODEL_NAME], env=OLLAMA_ENV)

CompletedProcess(args=['/usr/local/bin/ollama', 'run', 'qwen3.8-27b-uncensored-mtp'], returncode=0)

## 15. Test the public API from Ubuntu / phone
Quick Tunnels generate a random `*.trycloudflare.com` hostname for the current `cloudflared` process. The URL is temporary, so use the URL printed by the tunnel cell.

```bash
curl -sS \
  -H "Authorization: Bearer sk-mithu-local" \
  "https://YOUR-URL.trycloudflare.com/v1/models"

curl -sS \
  -H "Authorization: Bearer sk-mithu-local" \
  -H "Content-Type: application/json" \
  "https://YOUR-URL.trycloudflare.com/v1/chat/completions" \
  -d '{"model":"qwen3.8-27b-uncensored-mtp","messages":[{"role":"user","content":"Reply exactly: PUBLIC 256K API WORKS"}],"stream":false,"max_tokens":32}'
```

Note: the API is configured for `num_ctx=262144` (256K). A public smoke-test DNS error inside Kaggle does not change the model context configuration.


In [23]:
# ==============================================================
# QWEN3.8-27B UNCENSORED — 256K KAGGLE CHATBOT
#
# Run this cell AFTER your Ollama + OpenAI-compatible proxy are
# already running.
# ============================================================== 

import os
import sys
import json
import time
import subprocess
import urllib.request
import urllib.error
import socket

# ==============================================================
# 1. GRADIO
# ==============================================================

try:
    import gradio as gr
except ImportError:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-U",
        "gradio>=6,<7",
    ])
    import gradio as gr

print("Gradio version:", getattr(gr, "__version__", "unknown"))

# ==============================================================
# 2. CONFIG
# ============================================================== 

MODEL_NAME = globals().get(
    "MODEL_NAME",
    "qwen3.8-27b-uncensored-mtp",
)

API_KEY = globals().get(
    "API_KEY",
    os.environ.get("LLM_API_KEY", "sk-mithu-local"),
)

PROXY_URL = globals().get(
    "PROXY_URL",
    "http://127.0.0.1:18000",
).rstrip("/")

CONTEXT_LENGTH = 262144  # 256K

# ==============================================================
# 3. SHOW CONFIG
# ============================================================== 

print("=" * 72)
print("QWEN3.8-27B 256K CHATBOT")
print("=" * 72)
print("Model   :", MODEL_NAME)
print("Context :", f"{CONTEXT_LENGTH:,} tokens (256K)")
print("Backend :", PROXY_URL + "/v1")
print("=" * 72)

# ==============================================================
# 4. API CHECK
# ============================================================== 

def check_api():
    url = PROXY_URL + "/v1/models"
    req = urllib.request.Request(
        url,
        headers={
            "Authorization": "Bearer " + API_KEY,
            "Accept": "application/json",
        },
        method="GET",
    )

    try:
        with urllib.request.urlopen(req, timeout=30) as response:
            data = json.loads(response.read())

        print("OK: local OpenAI-compatible API is working")
        print("Available models:")

        for item in data.get("data", []):
            print("  -", item.get("id"))

        return True
    except urllib.error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="ignore")
        print("API HTTP error:", exc.code)
        print(body)
        return False
    except Exception as exc:
        print("API connection failed:", type(exc).__name__, exc)
        return False

if not check_api():
    raise RuntimeError(
        "Local proxy is not reachable. Start the Ollama/OpenAI proxy first."
    )

# ==============================================================
# 5. SYSTEM PROMPT
# ============================================================== 

DEFAULT_SYSTEM_PROMPT = (
    "You are Qwen3.8-27B running as a local AI assistant.\n\n"
    "Be accurate and practical. Be concise unless a detailed answer is requested.\n"
    "For programming questions, provide complete runnable code and explain important assumptions.\n"
    "For debugging, identify the root cause before proposing fixes.\n"
    "Do not invent APIs, commands, facts, or configuration options.\n"
    "Prefer production-quality solutions."
)

# ==============================================================
# 6. HISTORY NORMALIZATION
# ============================================================== 

def normalize_history(history):
    messages = []

    if not history:
        return messages

    for item in history:
        if not isinstance(item, dict):
            continue

        role = item.get("role")
        content = item.get("content", "")

        if role not in {"user", "assistant", "system"}:
            continue

        if isinstance(content, str):
            text_content = content
        elif isinstance(content, list):
            parts = []
            for part in content:
                if isinstance(part, str):
                    parts.append(part)
                elif isinstance(part, dict) and "text" in part:
                    parts.append(str(part["text"]))
            text_content = "\n".join(parts)
        else:
            text_content = str(content)

        if text_content.strip():
            messages.append({
                "role": role,
                "content": text_content,
            })

    return messages

# ==============================================================
# 7. QWEN REQUEST
# ============================================================== 

def ask_qwen(messages, temperature=0.7, max_tokens=1024):
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "stream": False,
        "temperature": float(temperature),
        "max_tokens": int(max_tokens),
        "options": {
            "num_ctx": CONTEXT_LENGTH,
        },
    }

    body = json.dumps(payload).encode("utf-8")

    req = urllib.request.Request(
        PROXY_URL + "/v1/chat/completions",
        data=body,
        headers={
            "Authorization": "Bearer " + API_KEY,
            "Content-Type": "application/json",
            "Accept": "application/json",
        },
        method="POST",
    )

    started = time.perf_counter()

    try:
        with urllib.request.urlopen(req, timeout=900) as response:
            raw = response.read()
    except urllib.error.HTTPError as exc:
        error_body = exc.read().decode("utf-8", errors="ignore")
        raise RuntimeError("HTTP {}\n{}".format(exc.code, error_body)) from exc
    except Exception as exc:
        raise RuntimeError("{}: {}".format(type(exc).__name__, exc)) from exc

    elapsed = time.perf_counter() - started

    try:
        result = json.loads(raw)
    except json.JSONDecodeError as exc:
        text_body = raw.decode("utf-8", errors="ignore")
        raise RuntimeError("API returned invalid JSON:\n" + text_body) from exc

    choices = result.get("choices") or []
    if not choices:
        raise RuntimeError(
            "API returned no choices:\n" + json.dumps(result, indent=2)
        )

    message = choices[0].get("message") or {}
    content = message.get("content", "")

    if not isinstance(content, str):
        content = str(content)

    usage = result.get("usage") or {}
    total_tokens = usage.get("total_tokens")

    return content.strip(), elapsed, total_tokens

# ==============================================================
# 8. CHAT CALLBACK
# ============================================================== 

def respond(message, history, system_prompt, temperature, max_tokens):
    if not message or not str(message).strip():
        return history or [], ""

    history = list(history or [])
    user_text = str(message).strip()

    model_messages = []

    model_messages.append({
        "role": "system",
        "content": (
            system_prompt.strip()
            if system_prompt and system_prompt.strip()
            else DEFAULT_SYSTEM_PROMPT
        ),
    })

    model_messages.extend(normalize_history(history))

    model_messages.append({
        "role": "user",
        "content": user_text,
    })

    try:
        answer, elapsed, total_tokens = ask_qwen(
            model_messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )

        if total_tokens is not None:
            footer = (
                "\n\n---\n"
                "*{:.1f}s · {:,} tokens · 256K context requested*"
            ).format(elapsed, total_tokens)
        else:
            footer = (
                "\n\n---\n"
                "*{:.1f}s · 256K context requested*"
            ).format(elapsed)

        answer += footer

    except Exception as exc:
        answer = (
            "❌ **Request failed**\n\n"
            "```text\n"
            + type(exc).__name__
            + ": "
            + str(exc)
            + "\n```"
        )

    history.append({
        "role": "user",
        "content": user_text,
    })
    history.append({
        "role": "assistant",
        "content": answer,
    })

    return history, ""

# ==============================================================
# 9. CLEAR
# ============================================================== 

def clear_chat():
    return [], ""

# ==============================================================
# 10. UI
# ============================================================== 

with gr.Blocks(title="Qwen3.8-27B 256K Chatbot") as demo:
    gr.Markdown(
        "# 🤖 Qwen3.8-27B Uncensored\n\n"
        "### Local 256K AI Chatbot\n\n"
        "**Model:** `" + MODEL_NAME + "`  \n"
        "**Context:** **262,144 tokens (256K)**  \n"
        "**Backend:** Ollama + OpenAI-compatible API"
    )

    chatbot = gr.Chatbot(
        type="messages",
        height=620,
        label="Qwen Chat",
        allow_tags=False,
    )

    with gr.Row():
        message = gr.Textbox(
            label="Message",
            placeholder="Ask Qwen anything...",
            lines=4,
            scale=8,
            autofocus=True,
        )
        send_button = gr.Button(
            "Send",
            variant="primary",
            scale=1,
        )

    with gr.Row():
        clear_button = gr.Button("🗑 Clear Chat")

    with gr.Accordion("⚙️ Settings", open=False):
        system_prompt = gr.Textbox(
            label="System Prompt",
            value=DEFAULT_SYSTEM_PROMPT,
            lines=10,
        )

        temperature = gr.Slider(
            minimum=0.0,
            maximum=2.0,
            value=0.7,
            step=0.05,
            label="Temperature",
        )

        max_tokens = gr.Slider(
            minimum=64,
            maximum=8192,
            value=1024,
            step=64,
            label="Maximum Output Tokens",
        )

    with gr.Accordion("ℹ️ Connection Information", open=False):
        gr.Markdown(
            "**Model:** `" + MODEL_NAME + "`\n\n"
            "**Endpoint:** `" + PROXY_URL + "/v1/chat/completions`\n\n"
            "**Context:** `262144` tokens (256K)\n\n"
            "Every request includes `num_ctx=262144`."
        )

    inputs = [
        message,
        chatbot,
        system_prompt,
        temperature,
        max_tokens,
    ]

    outputs = [
        chatbot,
        message,
    ]

    send_button.click(
        fn=respond,
        inputs=inputs,
        outputs=outputs,
    )

    message.submit(
        fn=respond,
        inputs=inputs,
        outputs=outputs,
    )

    clear_button.click(
        fn=clear_chat,
        inputs=None,
        outputs=outputs,
    )

# ==============================================================
# 11. LAUNCH
# ============================================================== 

print()
print("=" * 72)
print("STARTING QWEN3.8-27B 256K CHATBOT")
print("=" * 72)
print("Model   :", MODEL_NAME)
print("Context : 262,144 tokens (256K)")
print("Backend :", PROXY_URL + "/v1")
print("=" * 72)

# Public Gradio URL

# Find an available Gradio port instead of assuming 7860 is free.
def find_free_port(start=7860, end=7900):
    for port in range(start, end + 1):
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        try:
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            sock.bind(("0.0.0.0", port))
            return port
        except OSError:
            pass
        finally:
            sock.close()
    raise OSError(f"No free Gradio port found in {start}-{end}.")

GRADIO_PORT = find_free_port()
os.environ["GRADIO_SERVER_PORT"] = str(GRADIO_PORT)

print(f"Selected Gradio port: {GRADIO_PORT}")

demo.launch(
    server_name="0.0.0.0",
    server_port=GRADIO_PORT,
    share=True,
    show_error=True,
    inbrowser=False,
)


Gradio version: 5.50.0
QWEN3.8-27B 256K CHATBOT
Model   : qwen3.8-27b-uncensored-mtp
Context : 262,144 tokens (256K)
Backend : http://127.0.0.1:18000/v1
OK: local OpenAI-compatible API is working
Available models:
  - qwen3.8-27b-uncensored-mtp:latest
  - hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M


/tmp/ipykernel_58/1085207388.py:319: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(



STARTING QWEN3.8-27B 256K CHATBOT
Model   : qwen3.8-27b-uncensored-mtp
Context : 262,144 tokens (256K)
Backend : http://127.0.0.1:18000/v1
Selected Gradio port: 7861
* Running on local URL:  http://0.0.0.0:7861
* Running on public URL: https://60b1e05e99e2e3c05c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [27]:
# ============================================================
# KAGGLE QWEN3.8-27B 256K OPENAI-COMPATIBLE API PROXY
#
# Architecture:
#
#   Ubuntu / OpenAI SDK / UI
#             │
#             ▼
#        ngrok HTTPS
#             │
#             ▼
#    FastAPI Proxy (free port)
#             │
#             ▼
#      Ollama :11434
#             │
#             ▼
#   qwen3.8-27b-uncensored-mtp
#
# Context: 262,144 tokens (256K)
#
# Run this AFTER Ollama is already running in Kaggle.
# ============================================================

# ------------------------------------------------------------
# INSTALL DEPENDENCIES
# ------------------------------------------------------------
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "fastapi",
    "uvicorn",
    "httpx",
    "pyngrok",
])

# ------------------------------------------------------------
# IMPORTS
# ------------------------------------------------------------
import asyncio
import json
import os
import secrets
import socket
import threading
import time

import httpx
import uvicorn

from fastapi import FastAPI, Depends, HTTPException, Request
from fastapi.responses import Response, StreamingResponse
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok


# ============================================================
# CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# OLLAMA
# ------------------------------------------------------------

OLLAMA_HOST = "127.0.0.1"
OLLAMA_PORT = 11434

OLLAMA_BASE_URL = (
    f"http://{OLLAMA_HOST}:{OLLAMA_PORT}"
)

OLLAMA_OPENAI_URL = (
    f"{OLLAMA_BASE_URL}/v1"
)

# Your model
MODEL_NAME = os.getenv(
    "MODEL_NAME",
    "qwen3.8-27b-uncensored-mtp",
)

# Full requested context: 256K
CONTEXT_LENGTH = 262144

# Optional Ollama KV cache setting
KV_CACHE_TYPE = os.getenv(
    "OLLAMA_KV_CACHE_TYPE",
    "q8_0",
)


# ------------------------------------------------------------
# PROXY
# ------------------------------------------------------------

PROXY_HOST = "0.0.0.0"
PREFERRED_PROXY_PORT = 8080


# ------------------------------------------------------------
# API KEY
# ------------------------------------------------------------

# For this project the default is intentionally stable so you
# can reuse it from Ubuntu/OpenAI-compatible clients.
PROXY_API_KEY = os.getenv(
    "PROXY_API_KEY",
    "sk-mithu-local",
)


# ------------------------------------------------------------
# NGROK AUTH TOKEN
#
# SECURITY:
# Do NOT hard-code your ngrok token in this notebook.
#
# Recommended Kaggle secret:
#   NGROK_AUTH_TOKEN
# ------------------------------------------------------------

NGROK_AUTH_TOKEN = os.getenv(
    "NGROK_AUTH_TOKEN"
)

if not NGROK_AUTH_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient

        NGROK_AUTH_TOKEN = (
            UserSecretsClient()
            .get_secret("NGROK_AUTH_TOKEN")
        )
    except Exception:
        NGROK_AUTH_TOKEN = None

if not NGROK_AUTH_TOKEN:
    raise RuntimeError(
        "\n❌ NGROK_AUTH_TOKEN is missing.\n\n"
        "Add your ngrok auth token to Kaggle Secrets "
        "with the name:\n"
        "    NGROK_AUTH_TOKEN\n\n"
        "Do not paste the token directly into the notebook."
    )


# ============================================================
# PORT HELPERS
# ============================================================

def is_port_open(host: str, port: int) -> bool:
    """Return True when a TCP port is already listening."""

    sock = socket.socket(
        socket.AF_INET,
        socket.SOCK_STREAM,
    )

    sock.settimeout(0.5)

    try:
        return (
            sock.connect_ex(
                (host, port)
            ) == 0
        )

    finally:
        sock.close()


def find_free_port(
    start_port: int,
    end_port: int = 8090,
) -> int:
    """Find a free localhost TCP port."""

    for port in range(
        start_port,
        end_port + 1,
    ):
        if not is_port_open(
            "127.0.0.1",
            port,
        ):
            return port

    raise RuntimeError(
        f"No free proxy port found in "
        f"{start_port}-{end_port}."
    )


# ============================================================
# STEP 1 — CHECK OLLAMA
# ============================================================

print()
print("=" * 72)
print("🔍 CHECKING LOCAL OLLAMA")
print("=" * 72)

print(
    "Local OpenAI API:",
    f"{OLLAMA_OPENAI_URL}"
)

try:
    response = httpx.get(
        f"{OLLAMA_OPENAI_URL}/models",
        timeout=15.0,
    )
except Exception as exc:
    raise RuntimeError(
        "\n❌ Ollama is not reachable.\n\n"
        f"Expected:\n"
        f"{OLLAMA_OPENAI_URL}/models\n\n"
        f"Error:\n{exc}\n\n"
        "Make sure Ollama is running on port 11434."
    )


if response.status_code != 200:
    raise RuntimeError(
        "\n❌ Ollama returned an error.\n\n"
        f"HTTP status: {response.status_code}\n\n"
        f"Response:\n{response.text[:5000]}"
    )


# ============================================================
# STEP 2 — READ MODELS
# ============================================================

try:
    model_response = response.json()
except Exception as exc:
    raise RuntimeError(
        "\n❌ Ollama returned invalid JSON.\n\n"
        f"Error: {exc}\n\n"
        f"Raw response:\n{response.text[:5000]}"
    )


model_items = model_response.get(
    "data",
    [],
)

AVAILABLE_MODELS = []

for item in model_items:
    if isinstance(item, dict):
        model_id = item.get("id")
        if model_id:
            AVAILABLE_MODELS.append(
                str(model_id)
            )


if not AVAILABLE_MODELS:
    raise RuntimeError(
        "\n❌ No model IDs returned by Ollama.\n\n"
        + json.dumps(
            model_response,
            indent=2,
            ensure_ascii=False,
        )
    )


# Prefer the exact project model, then :latest,
# otherwise use the first returned model.
MODEL_CANDIDATES = [
    MODEL_NAME,
    f"{MODEL_NAME}:latest",
]

DEFAULT_MODEL = next(
    (
        candidate
        for candidate in MODEL_CANDIDATES
        if candidate in AVAILABLE_MODELS
    ),
    AVAILABLE_MODELS[0],
)


# ============================================================
# MODEL INFORMATION
# ============================================================

print()
print("=" * 72)
print("✅ LOCAL OLLAMA DETECTED")
print("=" * 72)

print()
print("🤖 AVAILABLE MODEL(S):")

for model in AVAILABLE_MODELS:
    print(f"   • {model}")

print()
print(
    f"🎯 DEFAULT MODEL: {DEFAULT_MODEL}"
)

print()
print(
    "🧠 CONTEXT:",
    f"{CONTEXT_LENGTH:,} tokens (256K)",
)

print()
print(
    "🔗 LOCAL API:",
    OLLAMA_OPENAI_URL,
)


# ============================================================
# STEP 3 — FASTAPI PROXY
# ============================================================

app = FastAPI(
    title="Kaggle Qwen3.8-27B 256K Proxy",
    version="2.0.0",
)


# ------------------------------------------------------------
# CORS
# ------------------------------------------------------------

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)


# ------------------------------------------------------------
# AUTHENTICATION
# ------------------------------------------------------------

security = HTTPBearer(
    auto_error=True
)


def verify_api_key(
    credentials: HTTPAuthorizationCredentials =
        Depends(security),
):
    if credentials.credentials != PROXY_API_KEY:
        raise HTTPException(
            status_code=401,
            detail="Invalid API key",
        )

    return credentials.credentials


# ============================================================
# ROOT
# ============================================================

@app.get("/")
async def root():
    return {
        "status": "ok",
        "service": (
            "Kaggle Qwen3.8-27B 256K "
            "OpenAI-compatible proxy"
        ),
        "architecture": (
            "ngrok -> FastAPI -> Ollama"
        ),
        "base_url": "/v1",
        "default_model": DEFAULT_MODEL,
        "models": AVAILABLE_MODELS,
        "context_length": CONTEXT_LENGTH,
        "context": "256K",
    }


# ============================================================
# HEALTH
# ============================================================

@app.get("/health")
async def health():
    ollama_online = False

    try:
        async with httpx.AsyncClient(
            timeout=5.0
        ) as client:
            result = await client.get(
                f"{OLLAMA_OPENAI_URL}/models"
            )

        ollama_online = (
            result.status_code == 200
        )

    except Exception:
        ollama_online = False

    return {
        "status": "ok",
        "proxy": "running",
        "ollama": (
            "online"
            if ollama_online
            else "offline"
        ),
        "local_api": OLLAMA_OPENAI_URL,
        "default_model": DEFAULT_MODEL,
        "models": AVAILABLE_MODELS,
        "context_length": CONTEXT_LENGTH,
        "context": "256K",
    }


# ============================================================
# CONFIG
# ============================================================

@app.get("/config")
async def config():
    return {
        "base_url": "/v1",
        "default_model": DEFAULT_MODEL,
        "models": AVAILABLE_MODELS,
        "context_length": CONTEXT_LENGTH,
        "context": "256K",
    }


# ============================================================
# OPENAI /v1/models
# ============================================================

@app.get("/v1/models")
async def models(
    _: str = Depends(verify_api_key),
):
    try:
        timeout = httpx.Timeout(
            connect=10.0,
            read=30.0,
            write=10.0,
            pool=10.0,
        )

        async with httpx.AsyncClient(
            timeout=timeout
        ) as client:
            upstream = await client.get(
                f"{OLLAMA_OPENAI_URL}/models"
            )

        content_type = (
            upstream.headers.get(
                "content-type",
                "application/json",
            )
        )

        return Response(
            content=upstream.content,
            status_code=upstream.status_code,
            media_type=content_type.split(";")[0],
        )

    except httpx.RequestError as exc:
        raise HTTPException(
            status_code=502,
            detail=(
                "Local Ollama unavailable: "
                f"{exc}"
            ),
        )


# ============================================================
# OPENAI /v1/chat/completions
# ============================================================

@app.post("/v1/chat/completions")
async def chat_completions(
    request: Request,
    _: str = Depends(verify_api_key),
):
    # --------------------------------------------------------
    # Read JSON
    # --------------------------------------------------------

    try:
        body = await request.json()
    except Exception:
        raise HTTPException(
            status_code=400,
            detail="Invalid JSON request body",
        )

    if not isinstance(body, dict):
        raise HTTPException(
            status_code=400,
            detail="JSON body must be an object",
        )


    # --------------------------------------------------------
    # Validate messages
    # --------------------------------------------------------

    messages = body.get("messages")

    if not isinstance(
        messages,
        list,
    ) or not messages:
        raise HTTPException(
            status_code=400,
            detail=(
                "`messages` must be a non-empty list."
            ),
        )


    # --------------------------------------------------------
    # Model selection
    # --------------------------------------------------------

    requested_model = body.get(
        "model"
    )

    if (
        not requested_model
        or str(requested_model).lower()
        in {
            "auto",
            "default",
        }
    ):
        body["model"] = DEFAULT_MODEL


    # --------------------------------------------------------
    # Force 256K context unless the caller explicitly requests
    # a smaller context.
    #
    # If you want strict 256K for every request, this can simply
    # be set to CONTEXT_LENGTH.
    # --------------------------------------------------------

    options = body.get(
        "options"
    )

    if not isinstance(
        options,
        dict,
    ):
        options = {}

    requested_context = options.get(
        "num_ctx"
    )

    if requested_context is None:
        options["num_ctx"] = CONTEXT_LENGTH
    else:
        try:
            requested_context = int(
                requested_context
            )
        except (TypeError, ValueError):
            raise HTTPException(
                status_code=400,
                detail="options.num_ctx must be an integer",
            )

        if requested_context <= 0:
            raise HTTPException(
                status_code=400,
                detail="options.num_ctx must be > 0",
            )

        # Do not silently exceed configured 256K.
        options["num_ctx"] = min(
            requested_context,
            CONTEXT_LENGTH,
        )

    body["options"] = options


    # --------------------------------------------------------
    # Keep model name compatible with Ollama.
    # --------------------------------------------------------

    # If the client uses qwen3.8-27b-uncensored-mtp,
    # prefer the exact available :latest tag when needed.
    requested_model = str(
        body.get("model")
    )

    if (
        requested_model
        not in AVAILABLE_MODELS
        and f"{requested_model}:latest"
        in AVAILABLE_MODELS
    ):
        body["model"] = (
            f"{requested_model}:latest"
        )


    # --------------------------------------------------------
    # STREAM FLAG
    # --------------------------------------------------------

    stream = bool(
        body.get(
            "stream",
            False,
        )
    )


    # ========================================================
    # STREAMING
    # ========================================================

    if stream:

        async def stream_generator():

            timeout = httpx.Timeout(
                connect=20.0,
                read=None,
                write=30.0,
                pool=20.0,
            )

            try:
                async with httpx.AsyncClient(
                    timeout=timeout
                ) as client:

                    async with client.stream(
                        "POST",
                        (
                            f"{OLLAMA_OPENAI_URL}"
                            "/chat/completions"
                        ),
                        json=body,
                        headers={
                            "Content-Type":
                                "application/json",
                            "Accept":
                                "text/event-stream",
                        },
                    ) as upstream:

                        if upstream.status_code >= 400:
                            error_content = (
                                await upstream.aread()
                            )

                            yield (
                                b"data: "
                                + error_content
                                + b"\n\n"
                            )

                            return


                        async for chunk in (
                            upstream.aiter_raw()
                        ):
                            if chunk:
                                yield chunk


            except Exception as exc:

                payload = {
                    "error": {
                        "message": str(exc),
                        "type": "proxy_error",
                    }
                }

                yield (
                    (
                        "data: "
                        + json.dumps(payload)
                        + "\n\n"
                    ).encode("utf-8")
                )


        return StreamingResponse(
            stream_generator(),
            media_type="text/event-stream",
            headers={
                "Cache-Control":
                    "no-cache, no-transform",
                "Connection":
                    "keep-alive",
                "X-Accel-Buffering":
                    "no",
            },
        )


    # ========================================================
    # NON-STREAMING
    # ========================================================

    try:

        timeout = httpx.Timeout(
            connect=20.0,
            read=900.0,
            write=30.0,
            pool=20.0,
        )

        async with httpx.AsyncClient(
            timeout=timeout
        ) as client:

            upstream = await client.post(
                (
                    f"{OLLAMA_OPENAI_URL}"
                    "/chat/completions"
                ),
                json=body,
                headers={
                    "Content-Type":
                        "application/json",
                    "Accept":
                        "application/json",
                },
            )

        content_type = (
            upstream.headers.get(
                "content-type",
                "application/json",
            )
        )

        return Response(
            content=upstream.content,
            status_code=upstream.status_code,
            media_type=content_type.split(";")[0],
        )

    except httpx.RequestError as exc:

        raise HTTPException(
            status_code=502,
            detail=(
                "Local Ollama error: "
                f"{exc}"
            ),
        )


# ============================================================
# STEP 4 — SELECT PROXY PORT
# ============================================================

print()
print("=" * 72)
print("🔧 SELECTING PROXY PORT")
print("=" * 72)

if is_port_open(
    "127.0.0.1",
    PREFERRED_PROXY_PORT,
):
    PROXY_PORT = find_free_port(
        PREFERRED_PROXY_PORT + 1
    )

    print(
        f"⚠️ Port {PREFERRED_PROXY_PORT} is busy."
    )

    print(
        f"✅ Using port {PROXY_PORT}"
    )

else:
    PROXY_PORT = PREFERRED_PROXY_PORT

    print(
        f"✅ Using port {PROXY_PORT}"
    )


# ============================================================
# STEP 5 — START FASTAPI
# ============================================================

print()
print("=" * 72)
print("🚀 STARTING FASTAPI")
print("=" * 72)

uvicorn_config = uvicorn.Config(
    app,
    host=PROXY_HOST,
    port=PROXY_PORT,
    log_level="info",
)

uvicorn_server = uvicorn.Server(
    uvicorn_config
)


def run_uvicorn():
    """
    Run Uvicorn on an independent event loop
    in a background thread.
    """

    loop = asyncio.new_event_loop()

    asyncio.set_event_loop(
        loop
    )

    try:
        loop.run_until_complete(
            uvicorn_server.serve()
        )
    finally:
        loop.close()


server_thread = threading.Thread(
    target=run_uvicorn,
    daemon=True,
    name="kaggle-qwen-fastapi",
)

server_thread.start()


# ============================================================
# STEP 6 — WAIT FOR FASTAPI
# ============================================================

print()
print(
    f"⏳ Waiting for FastAPI :{PROXY_PORT}..."
)

proxy_ready = False

for attempt in range(
    1,
    31,
):

    try:

        health = httpx.get(
            (
                f"http://127.0.0.1:"
                f"{PROXY_PORT}/health"
            ),
            timeout=2.0,
        )

        if health.status_code == 200:
            proxy_ready = True
            break

    except Exception:
        pass

    time.sleep(1)


if not proxy_ready:

    raise RuntimeError(
        "\n❌ FastAPI did not start.\n"
        f"Check port {PROXY_PORT} and notebook output."
    )


print(
    f"✅ FastAPI is ready on :{PROXY_PORT}"
)


# ============================================================
# STEP 7 — CONNECT NGROK
# ============================================================

print()
print("=" * 72)
print("🌍 STARTING NGROK")
print("=" * 72)

try:
    ngrok.set_auth_token(
        NGROK_AUTH_TOKEN
    )
except Exception as exc:
    raise RuntimeError(
        f"❌ ngrok authentication failed:\n{exc}"
    )


# Close tunnels from an earlier run
try:
    ngrok.kill()
except Exception:
    pass


try:
    tunnel = ngrok.connect(
        addr=PROXY_PORT,
        proto="http",
    )
except Exception as exc:
    raise RuntimeError(
        f"❌ Failed to create ngrok tunnel:\n{exc}"
    )


PUBLIC_URL = (
    tunnel.public_url
    .rstrip("/")
)

BASE_URL = (
    f"{PUBLIC_URL}/v1"
)


# ============================================================
# STEP 8 — TEST LOCAL PROXY
# ============================================================

print()
print("=" * 72)
print("🧪 TESTING LOCAL PROXY")
print("=" * 72)

try:

    local_health = httpx.get(
        (
            f"http://127.0.0.1:"
            f"{PROXY_PORT}/health"
        ),
        timeout=10,
    )

    if local_health.status_code == 200:

        print(
            "✅ Local health:",
            local_health.json(),
        )

    else:

        print(
            "⚠️ Local health returned:",
            local_health.status_code,
        )

except Exception as exc:

    print(
        "⚠️ Local proxy test failed:",
        exc,
    )


# ============================================================
# STEP 9 — TEST PUBLIC PROXY
# ============================================================

print()
print("=" * 72)
print("🧪 TESTING PUBLIC PROXY")
print("=" * 72)

public_health_ok = False

try:

    public_headers = {
        "Authorization":
            f"Bearer {PROXY_API_KEY}",
    }

    public_response = httpx.get(
        f"{PUBLIC_URL}/health",
        headers=public_headers,
        timeout=30.0,
    )

    if public_response.status_code == 200:

        public_health_ok = True

        print(
            "✅ Public health:",
            public_response.json(),
        )

    else:

        print(
            "⚠️ Public health returned:",
            f"HTTP {public_response.status_code}",
        )

        print(
            public_response.text[:3000]
        )

except Exception as exc:

    print(
        "⚠️ Public health test failed:",
        exc,
    )


# ============================================================
# STEP 10 — FINAL API INFORMATION
# ============================================================

print()
print("=" * 72)

if public_health_ok:
    print(
        "🚀 QWEN3.8-27B 256K API IS READY"
    )
else:
    print(
        "⚠️ PROXY IS RUNNING, "
        "BUT PUBLIC HEALTH TEST FAILED"
    )

print("=" * 72)

print()
print("🧠 ARCHITECTURE")
print("-" * 72)
print("Client / UI")
print("     ↓")
print("ngrok HTTPS")
print("     ↓")
print(
    f"FastAPI :{PROXY_PORT}"
)
print("     ↓")
print("Ollama :11434")
print("     ↓")
print(
    f"Qwen model: {DEFAULT_MODEL}"
)
print()
print(
    "Context:",
    f"{CONTEXT_LENGTH:,} tokens (256K)",
)

print()
print("🌍 OPENAI-COMPATIBLE")
print("-" * 72)
print(
    f"BASE URL: {BASE_URL}"
)

print()
print("🔑 API KEY")
print("-" * 72)
print(
    PROXY_API_KEY
)

print()
print("🤖 MODEL")
print("-" * 72)
print(
    DEFAULT_MODEL
)

print()
print("❤️ HEALTH")
print("-" * 72)
print(
    f"{PUBLIC_URL}/health"
)

print()
print("🤖 MODELS")
print("-" * 72)
print(
    f"{BASE_URL}/models"
)

print()
print("💬 CHAT")
print("-" * 72)
print(
    f"{BASE_URL}/chat/completions"
)

print()
print("=" * 72)
print("📌 COPY THESE INTO YOUR OPENAI-COMPATIBLE CLIENT")
print("=" * 72)

print()
print(
    f"Base URL : {BASE_URL}"
)

print(
    f"API Key  : {PROXY_API_KEY}"
)

print(
    f"Model    : {DEFAULT_MODEL}"
)

print()
print(
    "Python OpenAI SDK:"
)

print(
    "from openai import OpenAI"
)

print(
    "client = OpenAI("
)

print(
    f'    base_url="{BASE_URL}",'
)

print(
    f'    api_key="{PROXY_API_KEY}",'
)

print(")")

print()
print(
    "Example request:"
)

print(
    "response = client.chat.completions.create("
)

print(
    f'    model="{DEFAULT_MODEL}",'
)

print(
    '    messages=[{"role":"user","content":"Hello"}],'
)

print(
    '    stream=False,'
)

print(
    '    max_tokens=512,'
)

print(
    '    extra_body={"options":{"num_ctx":262144}},'
)

print(")")

print()
print("=" * 72)
print("✅ Ollama is local.")
print("✅ FastAPI is running in a background thread.")
print("✅ ngrok is connected.")
print("✅ Public API is OpenAI-compatible.")
print("✅ 256K context is requested by default.")
print("⚠️ Keep the Kaggle session running.")
print("=" * 72)



🔍 CHECKING LOCAL OLLAMA
Local OpenAI API: http://127.0.0.1:11434/v1

✅ LOCAL OLLAMA DETECTED

🤖 AVAILABLE MODEL(S):
   • qwen3.8-27b-uncensored-mtp:latest
   • hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M

🎯 DEFAULT MODEL: qwen3.8-27b-uncensored-mtp:latest

🧠 CONTEXT: 262,144 tokens (256K)

🔗 LOCAL API: http://127.0.0.1:11434/v1

🔧 SELECTING PROXY PORT
⚠️ Port 8080 is busy.
✅ Using port 8081

🚀 STARTING FASTAPI


INFO:     Started server process [58]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8081 (Press CTRL+C to quit)



⏳ Waiting for FastAPI :8081...
INFO:     127.0.0.1:59752 - "GET /health HTTP/1.1" 200 OK
✅ FastAPI is ready on :8081

🌍 STARTING NGROK

🧪 TESTING LOCAL PROXY
INFO:     127.0.0.1:59756 - "GET /health HTTP/1.1" 200 OK
✅ Local health: {'status': 'ok', 'proxy': 'running', 'ollama': 'online', 'local_api': 'http://127.0.0.1:11434/v1', 'default_model': 'qwen3.8-27b-uncensored-mtp:latest', 'models': ['qwen3.8-27b-uncensored-mtp:latest', 'hf.co/JonathanColetti/Qwen3.8-27B-Uncensored-GGUF:Q4_K_M'], 'context_length': 262144, 'context': '256K'}

🧪 TESTING PUBLIC PROXY
⚠️ Public health returned: HTTP 400
<!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-s